Classical SVC over 25 tasks using majority vote decision rule

Importing the libraries

In [ ]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import mode

Importing the dataset

In [ ]:
df=pd.read_csv("data.csv")
df

,ID,air_time1,disp_index1,gmrt_in_air1,gmrt_on_paper1,max_x_extension1,max_y_extension1,mean_acc_in_air1,mean_acc_on_paper1,mean_gmrt1,...,mean_jerk_in_air25,mean_jerk_on_paper25,mean_speed_in_air25,mean_speed_on_paper25,num_of_pendown25,paper_time25,pressure_mean25,pressure_var25,total_time25,class
0,id_1,5160,0.000013,120.804174,86.853334,957,6601,0.361800,0.217459,103.828754,...,0.141434,0.024471,5.596487,3.184589,71,40120,1749.278166,296102.7676,144605,P
1,id_2,51980,0.000016,115.318238,83.448681,1694,6998,0.272513,0.144880,99.383459,...,0.049663,0.018368,1.665973,0.950249,129,126700,1504.768272,278744.2850,298640,P
2,id_3,2600,0.000010,229.933997,172.761858,2333,5802,0.387020,0.181342,201.347928,...,0.178194,0.017174,4.000781,2.392521,74,45480,1431.443492,144411.7055,79025,P
3,id_4,2130,0.000010,369.403342,183.193104,1756,8159,0.556879,0.164502,276.298223,...,0.113905,0.019860,4.206746,1.613522,123,67945,1465.843329,230184.7154,181220,P
4,id_5,2310,0.000007,257.997131,111.275889,987,4732,0.266077,0.145104,184.636510,...,0.121782,0.020872,3.319036,1.680629,92,37285,1841.702561,158290.0255,72575,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,id_170,2930,0.000010,241.736477,176.115957,1839,6439,0.253347,0.174663,208.926217,...,0.119152,0.020909,4.508709,2.233198,96,44545,1798.923336,247448.3108,80335,H
170,id_171,2140,0.000009,274.728964,234.495802,2053,8487,0.225537,0.174920,254.612383,...,0.174495,0.017640,4.685573,2.806888,84,37560,1725.619941,160664.6464,345835,H
171,id_172,3830,0.000008,151.536989,171.104693,1287,7352,0.165480,0.161058,161.320841,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,H
172,id_173,1760,0.000008,289.518195,196.411138,1674,6946,0.518937,0.202613,242.964666,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,H


y label

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=df.iloc[:,-1].values #get the y labels
y=le.fit_transform(y)

An array containing all 25 tasks

In [ ]:
X_tasks=[] #list of 25 arrays each having 18 features
for t in range(1, 26):
    cols_t = [f"air_time{t}",
              f"disp_index{t}",
              f"gmrt_in_air{t}",
              f"gmrt_on_paper{t}",
              f"max_x_extension{t}",
              f"max_y_extension{t}",
              f"mean_acc_in_air{t}",
              f"mean_acc_on_paper{t}",
              f"mean_gmrt{t}",
              f"mean_jerk_in_air{t}",
              f"mean_jerk_on_paper{t}",
              f"mean_speed_in_air{t}",
              f"mean_speed_on_paper{t}",
              f"num_of_pendown{t}",
              f"paper_time{t}",
              f"pressure_mean{t}",
              f"pressure_var{t}",
              f"total_time{t}"]
    X_tasks.append(df[cols_t].values)

Classical SVM on 25 tasks

In [ ]:
from sklearn.model_selection import GridSearchCV,ShuffleSplit,train_test_split #for hyperparameter search
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import numpy as np
outer_cv=ShuffleSplit(n_splits=20,test_size=0.2,random_state=19)
outer_splits = list(outer_cv.split(X_tasks[0], y))
pipe_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42))
])
param_grid_svc={'svc__kernel':['linear','poly','rbf','sigmoid'],
                'svc__C':[0.1,1,10,100],
                'svc__gamma':['scale',1,0.1,0.01,0.001],
                'svc__tol':[1e-3,1e-4,1e-5]}
all_task_accs=[] #stores accuracy per task per split
all_task_preds=[] #stores predictions per task per split
for task_id,X in enumerate(X_tasks):
  acc_svc_tuned=[] #20 accuracies per task
  task_preds=[] #20 array of prediction per task
  for i,(trainval_idx,test_idx) in enumerate(outer_splits):
      X_trainval,X_test=X[trainval_idx],X[test_idx]
      y_trainval,y_test=y[trainval_idx],y[test_idx]
      X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,stratify=y_trainval,random_state=42)# further splitting
      grid_svc=GridSearchCV( estimator=pipe_svc, #model to tune
                            param_grid=param_grid_svc, # parameters to try
                            cv=3, #no of folds
                            scoring='accuracy', #how to measure the performace
                            n_jobs=-1 #all cpu cores to speed the training
                          ) #grid search on only train vs val
      grid_svc.fit(X_train,y_train)
      best_model=grid_svc.best_estimator_
      best_model.fit(np.vstack([X_train,X_val]),np.concatenate([y_train,y_val])) #joins two part to form train+val set
      #predictions
      y_pred=best_model.predict(X_test)
      task_preds.append(y_pred)
      acc_svc_tuned.append(np.mean(y_pred==y_test)) #list of 20 accuracies per task
  all_task_accs.append(acc_svc_tuned) #list of 25 lists each having 20 accuracies
  all_task_preds.append(task_preds) #list of 25 lists each having 20 predictions
all_task_accs=np.array(all_task_accs)
all_task_preds=np.array(all_task_preds,dtype=object) #predictions
task_means=all_task_accs.mean(axis=1)*100 #mean accuracy of each task across 20 splits
task_stds=all_task_accs.std(axis=1)*100 #std accuracy of each task across 20 splits
#print per-task accuracies
for t in range(25):
    print(f"Task {t+1}: {task_means[t]:.2f} ± {task_stds[t]:.2f}")
#compute majority vote accuracies across all 25 tasks per outer split
majority_accs=[] #[acc_split1, acc_split2, ..., acc_split20]
n_splits=outer_cv.get_n_splits() #no of outer splits
#store test indices once
for split_idx in range(n_splits): #prediction of all tasks for this split
  split_preds=[all_task_preds[t][split_idx] for t in range(25)]#collect predictions for all 25 tasks(for this split)
  split_preds=np.array(split_preds,dtype=int)
  #majority vote
  maj_vote,_=mode(split_preds,axis=0) # _ ignore the counts, mode=most frequent value
  maj_vote=maj_vote.ravel() #flatten it to make compatible to y_test (multi-dimensional array to 1D array)
  #true label for one of the tasks
  _,test_idx=outer_splits[split_idx]
  y_test=y[test_idx]
  #accuracies
  majority_accs.append(np.mean(maj_vote==y_test))
majority_accs=np.array(majority_accs)
mean_majority=majority_accs.mean()*100
std_majority=majority_accs.std()*100
print(f"\nMajority vote accuracy (all 25 tasks): {mean_majority:.2f} ± {std_majority:.2f}")





Task 1: 63.71 ± 6.46
Task 2: 66.86 ± 8.15
Task 3: 65.71 ± 6.39
Task 4: 67.43 ± 8.25
Task 5: 67.86 ± 6.87
Task 6: 67.43 ± 9.91
Task 7: 73.43 ± 7.98
Task 8: 71.86 ± 5.13
Task 9: 73.29 ± 8.63
Task 10: 64.43 ± 8.50
Task 11: 61.14 ± 11.15
Task 12: 65.43 ± 7.82
Task 13: 66.00 ± 7.11
Task 14: 66.14 ± 8.54
Task 15: 72.71 ± 7.37
Task 16: 69.71 ± 7.36
Task 17: 70.86 ± 8.21
Task 18: 65.29 ± 6.96
Task 19: 69.43 ± 8.95
Task 20: 69.43 ± 8.58
Task 21: 70.57 ± 7.46
Task 22: 68.00 ± 5.39
Task 23: 75.00 ± 5.56
Task 24: 71.29 ± 6.54
Task 25: 71.71 ± 6.45

Majority vote accuracy (all 25 tasks): 85.57 ± 7.37


classical svm on 5 best tasks

In [ ]:

#compute majority vote accuracies for 5 best tasks per outer split
majority_accs=[] #[acc_split1, acc_split2, ..., acc_split20]
n_splits=outer_cv.get_n_splits() #no of outer splits
#store test indices once
best_task_indices=np.argsort(task_means)[::-1][:5]
print("\nBest 5 tasks:")
for t in best_task_indices:
    print(f"Task {t+1}: {task_means[t]:.2f} ± {task_stds[t]:.2f}")
for split_idx in range(n_splits): #prediction of all tasks for this split
  split_preds=[all_task_preds[t][split_idx] for t in best_task_indices]
  split_preds=np.array(split_preds,dtype=int)
  #majority vote
  maj_vote,_=mode(split_preds,axis=0) # _ ignore the counts, mode=most frequent value
  maj_vote=maj_vote.ravel() #flatten it to make compatible to y_test (multi-dimensional array to 1D array)
  #true label for one of the tasks
  _,test_idx=outer_splits[split_idx]
  y_test=y[test_idx]
  #accuracies
  majority_accs.append(np.mean(maj_vote==y_test))
majority_accs=np.array(majority_accs)
mean_majority=majority_accs.mean()*100
std_majority=majority_accs.std()*100
print(f"Majority vote accuracy (best 5 tasks): {mean_majority:.2f} ± {std_majority:.2f}")






Best 5 tasks:
Task 23: 75.00 ± 5.56
Task 7: 73.43 ± 7.98
Task 9: 73.29 ± 8.63
Task 15: 72.71 ± 7.37
Task 8: 71.86 ± 5.13
Majority vote accuracy (best 5 tasks): 81.29 ± 7.31
